In [ ]:
"""
MLP MNIST 训练脚本

训练流程:
1. 加载 MNIST 数据集 (自动下载)
2. 定义数据预处理 (ToTensor + Normalize)
3. 创建 DataLoader
4. 定义模型、损失函数、优化器
5. 训练循环: 前向传播 -> 计算损失 -> 反向传播 -> 更新参数
6. 验证循环: 不计算梯度，评估准确率
7. 保存模型权重和 Loss 曲线
"""
import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')  # 无GUI环境下使用
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



def get_args():
    parser = argparse.ArgumentParser(description='MLP MNIST Training')
    parser.add_argument('--epochs', type=int, default=10, help='训练轮数')
    parser.add_argument('--batch-size', type=int, default=64, help='批大小')
    parser.add_argument('--lr', type=float, default=0.001, help='学习率')
    parser.add_argument('--hidden1', type=int, default=256, help='第一隐藏层维度')
    parser.add_argument('--hidden2', type=int, default=128, help='第二隐藏层维度')
    parser.add_argument('--dropout', type=float, default=0.2, help='Dropout概率')
    parser.add_argument('--data-dir', type=str, default='./data', help='数据存放目录')
    parser.add_argument('--save-dir', type=str, default='./checkpoints', help='模型保存目录')
    parser.add_argument('--seed', type=int, default=42, help='随机种子')
    return parser.parse_known_args()[0]



def set_seed(seed):
    """设置随机种子，保证实验可复现"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data_loaders(data_dir, batch_size):
    """
    加载 MNIST 数据集

    数据预处理说明:
    - ToTensor(): 将 PIL 图片 (0-255) 转为 Tensor (0-1)，形状从 [H,W] 变为 [1,H,W]
    - Normalize((0.1307,), (0.3081,)): 用 MNIST 数据集的全局均值和标准差做标准化
      均值 0.1307，标准差 0.3081 是 MNIST 训练集统计得到的固定值
      标准化后数据分布接近 N(0,1)，有助于梯度下降收敛
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # train=True 加载训练集 (60000张)，train=False 加载测试集 (10000张)
    train_dataset = datasets.MNIST(
        root=data_dir, train=True, download=True, transform=transform
    )
    test_dataset = datasets.MNIST(
        root=data_dir, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )

    return train_loader, test_loader


def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()  # 设置为训练模式 (Dropout 生效)
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # 1. 前向传播: 输入图片 -> 输出 logits
        output = model(data)

        # 2. 计算损失: CrossEntropyLoss = LogSoftmax + NLLLoss
        loss = criterion(output, target)

        # 3. 反向传播前清空梯度 (PyTorch 默认梯度累加)
        optimizer.zero_grad()

        # 4. 反向传播: 自动计算 loss 对每个参数的梯度
        loss.backward()

        # 5. 更新参数: optimizer 根据梯度和学习率更新权重
        optimizer.step()

        # 统计
        running_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


@torch.no_grad()  # 验证时不需要计算梯度，节省内存和加速
def evaluate(model, test_loader, criterion, device):
    """验证/测试"""
    model.eval()  # 设置为评估模式 (Dropout 关闭)
    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)

        running_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    avg_loss = running_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


def plot_curves(train_losses, test_losses, train_accs, test_accs, save_path):
    """绘制 Loss 和 Accuracy 曲线"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    epochs = range(1, len(train_losses) + 1)

    # Loss 曲线
    ax1.plot(epochs, train_losses, 'b-', label='Train Loss')
    ax1.plot(epochs, test_losses, 'r-', label='Test Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Test Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy 曲线
    ax2.plot(epochs, train_accs, 'b-', label='Train Accuracy')
    ax2.plot(epochs, test_accs, 'r-', label='Test Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Test Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"曲线已保存到: {save_path}")


def main():
    args = get_args()
    set_seed(args.seed)

    # 自动选择设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # 数据
    train_loader, test_loader = get_data_loaders(args.data_dir, args.batch_size)
    print(f"训练集: {len(train_loader.dataset)} 张, 测试集: {len(test_loader.dataset)} 张")

    # 模型
    model = MLP(
        hidden1=args.hidden1,
        hidden2=args.hidden2,
        dropout=args.dropout
    ).to(device)
    print(f"模型参数量: {count_parameters(model):,}")

    # 损失函数: 交叉熵损失，适合多分类任务
    # 内部会先做 LogSoftmax，所以模型最后一层不需要加 Softmax
    criterion = nn.CrossEntropyLoss()

    # 优化器: Adam，自适应学习率，比 SGD 收敛更快
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    # 创建保存目录
    os.makedirs(args.save_dir, exist_ok=True)

    # 训练循环
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    best_acc = 0.0

    print(f"\n开始训练，共 {args.epochs} 个 epoch...")
    print("-" * 70)

    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)

        print(f"Epoch {epoch:2d}/{args.epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.2f}%")

        # 保存最佳模型
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc,
                'args': vars(args),
            }, os.path.join(args.save_dir, 'mlp_mnist_best.pth'))

    print("-" * 70)
    print(f"训练完成! 最佳测试准确率: {best_acc:.2f}%")

    # 保存最后一个 epoch 的模型
    torch.save({
        'epoch': args.epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'test_acc': test_acc,
        'args': vars(args),
    }, os.path.join(args.save_dir, 'mlp_mnist_last.pth'))

    # 绘制并保存曲线
    plot_curves(
        train_losses, test_losses, train_accs, test_accs,
        os.path.join(args.save_dir, 'training_curves.png')
    )

    # 保存训练记录
    with open(os.path.join(args.save_dir, 'training_log.txt'), 'w') as f:
        f.write("MLP MNIST Training Log\n")
        f.write("=" * 50 + "\n")
        f.write(f"参数量: {count_parameters(model):,}\n")
        f.write(f"最佳测试准确率: {best_acc:.2f}%\n")
        f.write(f"超参数: {vars(args)}\n\n")
        f.write("Epoch,Train_Loss,Train_Acc,Test_Loss,Test_Acc\n")
        for i in range(len(train_losses)):
            f.write(f"{i+1},{train_losses[i]:.4f},{train_accs[i]:.2f},{test_losses[i]:.4f},{test_accs[i]:.2f}\n")


if __name__ == "__main__":
    main()


使用设备: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [08:43<00:00, 18.9kB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 117kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



 66%|██████▌   | 1.08M/1.65M [13:36:38<41:19, 229B/s]